# Diffusion Transformer(DiT)
기존 Diffusion 모델에서는, U-Net을 활용해 각 스텝에서 추가된 노이즈(정확하게는 그 스텝에서의 노이즈를 예측) $\epsilon$을 예측하고 제거하는 방식으로 이미지를 생성합니다.

하지만, U-Net은 굉장히 오래된 모델 구조이며, 명시적으로 어텐션 레이어를 중간중간 집어넣어 생성된 픽셀 간 관계도를 제공치 않는다면 썩 마음에 드는 결과가 나오지 않는다.

즉, 어짜피 어텐션을 사용할 것이면 그냥 트랜스포머를 활용하면 되는 것 아닌가? 라는 느낌으로 출발했다고 생각할 수 있다.

## Preliminary - ViT
DiT의 구조는 매우 많이 이전 ViT에서 가져온다. 각 픽셀 단위를 먹이게 된다면 너무나도 토큰 시퀀스가 커지기 때문에, 이를 처리하기 위해 패치 단위로 이미지를 압축시켜(stride와 kernel size가 똑같은 CNN) 먹이는 방식이다.

위치 임베딩은 2차원 포지셔널 임베딩을 활용하며, 고정으로 해도 되고 leranable하게 가져가도 된다.

## DiT와 ViT의 차이점

우선 꼭 필요한 타임스텝에 대한 정보를 인코딩하여 넘겨줘야 하고, 클래스에 대한 임베딩 또한 존재해야 합니다. 이 둘은 AdaLN(Adaptive Layer Normalization)을 활용하여 LayerNorm 이후에 바로 조건적인 것을 shift하여 집어넣는 방식으로 동작됩니다.

그 밖에는 일단 논문 상 구조에 따르면 VAE를 활용해 압축된 latent representation을 가지고 DiT를 적용한다는 점이 있겠다.


In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

D:\리서치\스터디 리드\TorchLeet\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [40]:
def timestep_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, dtype=t.dtype, device=t.device) / half)
    args = t[:, None] * freqs[None]
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2: # Should raise an error tbh
        emb = F.pad(emb, (0, 1))
    return emb


In [41]:
class PatchEmbed(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int, patch_size: int):
        super().__init__()
        self.patcher = nn.Conv2d(in_channels=in_ch, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)
        self.patch_size = patch_size

    def forward(self, x: torch.Tensor):
        x = self.patcher(x) # 이거 해도 B, C, H, W임, 단지 H, W 차원 줄음
        h = x.size(2) # 한 변에 몇개 패치 있는지 추후 사용해서 이미지 재구성하기 위해 반환합니다
        # 64x64 에 patchsize 4이면 16x16. h=16
        x = x.flatten(2).transpose(1, 2) # B, N, D로 변환해주기. 추후 rearrange로 하자!

        return x, h


In [42]:
class ClassEmbed(nn.Module):
    def __init__(self, num_classes: int, embed_dim: int):
        super().__init__()
        self.emb = nn.Embedding(num_classes, embed_dim)
    def forward(self, c: torch.Tensor):
        return self.emb(c)

In [43]:
class FiLM(nn.Module):
    # 이걸 활용해 추가 임베딩을 정규화 뒤에 shift&scale하여 적용시켜줍니다.
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim * 2)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias) # AdaLN-Zero는 0으로 초기화

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        scale, shift = self.proj(cond).chunk(2, dim=-1)
        print(x.shape, scale.shape, shift.shape)
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


In [44]:
class DiTBlock(nn.Module):
    # 이제 모든 것을 합쳐봅시다
    def __init__(self, dim: int, heads: int = 8, mlp_ratio: int = 4, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.RMSNorm(dim)
        self.norm2 = nn.RMSNorm(dim)
        self.film1 = FiLM(dim)
        self.film2 = FiLM(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True) # 왜 배치가 기본으로 첫째 차원이 아닌지 의문이 드네요
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        h = self.film1(self.norm1(x), cond)
        h, _ = self.attn(h, h, h, need_weights = False) # 나중에는 다 Linear 만들어야 하는데 귀찮아서 ㅎㅎ
        x = x + h
        h = self.film2(self.norm2(x), cond)
        x = x + self.mlp(h)
        return x


In [45]:
class DiT(nn.Module):
    def __init__(
        self,
        img_size: int = 64,
        patch_size: int = 2,
        in_channels: int = 4,
        out_channels: int = 4,
        width: int = 728,
        depth: int = 12,
        heads: int = 8,
        num_classes: int = 0,
    ):
        super().__init__()
        assert img_size % patch_size == 0, "패치는 남는 것이 없어야 합니다. 차원이 나눠져야 함"
        num_patches = (img_size // patch_size) ** 2
        self.width = width

        self.patch_embed = PatchEmbed(in_channels, width, patch_size)
        self.class_embed = ClassEmbed(num_classes, width) if num_classes > 0 else None

        # Learnable pos emb
        self.pos = nn.Parameter(torch.randn(1, num_patches, width) / math.sqrt(width))

        # Time embedding → width dims
        time_dim = width * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(width, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, width),
        )

        self.blocks = nn.ModuleList([DiTBlock(width, heads=heads) for _ in range(depth)])

        self.norm_f = nn.RMSNorm(width)
        self.unpatch = nn.ConvTranspose2d(width, out_channels, patch_size, patch_size)
    def forward(
        self,
        x: torch.Tensor, # 노이즈를 먹인 latent (B,C,H,W)
        t: torch.Tensor, # Timestep conditional
        y: torch.Tensor = None,
    ) -> torch.Tensor:
        B = x.size(0)

        x, side = self.patch_embed(x) # B N D랑 한 변의 길이
        x = x + self.pos

        cond = self.time_mlp(timestep_embedding(t, self.width))
        if y is not None:
            cond = cond + self.class_embed(y)

        for blk in self.blocks:
            x = blk(x, cond)
        x = self.norm_f(x)

        # 이미지 모양으로 다시 돌려두기
        x = x.transpose(1, 2).reshape(B, -1, side, side)
        return self.unpatch(x)

In [46]:
model = DiT(num_classes=1000)
z = torch.randn(2, 4, 64, 64)
t = torch.randint(0, 1000, (2,))
y = torch.randint(0, 1000, (2,))
print("output", model(z, t, y).shape)  # (2, 4, 64, 64)

torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2, 1024, 728]) torch.Size([2, 728]) torch.Size([2, 728])
torch.Size([2